## TOOLS
### Models can  request to call tools that  performs  tasks such as fetching data  from database,searching the web ,or running code.Tools are pairing of 
 - A schema ,including the name of tool, a decription ,and /or aargument defination(often a JSON Schema) 
 - A function or continue to execute 

In [3]:
import os 
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-20b")
responce = model.invoke("why  do parrot do talk ")
responce.content

'**Short answer:**  \nParrots don’t “talk” the way humans do. They mimic sounds—including human speech—because it’s part of their natural communication system, a sign of their intelligence, and a useful tool for social interaction and survival.\n\n---\n\n### 1.  Why parrots mimic at all  \n| Reason | What it looks like in nature | Why it matters |\n|--------|------------------------------|----------------|\n| **Social bonding** | Parrots live in flocks and use calls to keep groups together. | Mimicking a familiar voice or call strengthens bonds with flock mates or a human “family.” |\n| **Territorial signaling** | They repeat alarm calls or predator sounds. | Repeating a predator’s call can warn others or deter the threat. |\n| **Resource location** | Some species copy the call of a successful forager. | Mimicking can help the whole group find food. |\n| **Curiosity & learning** | Young parrots experiment with sounds to see what’s possible. | It’s a natural part of vocal development. |

In [6]:
from langchain.tools import tool

@tool
def get_wther(location:str)->str:
    """Get the wether at a location"""
    return f"its suuny at {location}"
    
model_with_tools = model.bind_tools([get_wther])


# we can also use this way for this 
# from pydantic._internal import _core_metadata
# from langchain.agents import create_agent

# def getWether(city:str) ->str:
#     """Get the wether for city"""
#     return f"the wether in {city} is sunny "
# agent = create_agent(
#     model ="groq:llama-3.3-70b-versatile",
#     tools=[getWether],
#     system_prompt="you are a helpfull asistant"
# )
# agent

In [8]:
#now after binding  how we call this tool
responce = model_with_tools.invoke("what is the wether in boston ?")
print(responce)
for tool_call in responce.tool_calls:
    # view  tool calls made by model 
    print(f"tool:{tool_call['name']}")
    print(f"Args:{tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'User asks: "what is the wether in boston ?" Likely they mean weather. Need to use function get_wther.', 'tool_calls': [{'id': 'fc_2b42e8c4-bff9-402c-ba98-f881f4768b8f', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_wther'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 131, 'total_tokens': 183, 'completion_time': 0.057450909, 'completion_tokens_details': {'reasoning_tokens': 28}, 'prompt_time': 0.007493752, 'prompt_tokens_details': None, 'queue_time': 0.376212649, 'total_time': 0.064944661}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e23fc997ca', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a052eb-1448-73d2-b5fb-5695ab45df42-0' tool_calls=[{'name': 'get_wther', 'args': {'location': 'Boston'}, 'id': 'fc_2b42e8c4-bff9-402c-ba98-f881f4768b8f', 'type': 'tool_call'}] invalid_tool

### Tool execution loop 

In [ ]:
from langchain_core.prompts import message
# step 1: Model generates tool calls
message = [{"role":"user","content":"what's the wether in boston "}]
ai_msg = model_with_tools.invoke(message)
message.append(ai_msg)

# step 2: Execute the tools and collect results
for tool_call in ai_msg.tool_calls:
    # execute the tool with  the genrated  arguments
    tool_result =  get_wther.invoke(tool_call)
    message.append(tool_result)
#step 3 : pass results baack to model for final responce 
final_responce = model_with_tools.invoke(message)
print(final_responce.text)
# the current wehther in boston is 72F and  sunny 